# Extraction of medications on admission from clinical notes

## Extraction run over the experiment grid

- Each train/val note is processed 9 times: 
    -  3 models x 3 prompting strategies.

- All input variables (paths, prompt files, grid, Ollama runtime settings) are defined HERE and passed as parameters. utils/extraction.py holds pure logic only.

- Each train/val note is processed once per (model, strategy) cell. 

- Model on the OUTER loop avoids repeated model load/unload in Ollama VRAM.

- Caching in extract_note() makes this safely re-runnable after
interruption without repeating completed calls.



## Imports

In [ ]:
import json
import shutil
from pathlib import Path
from datetime import date
 
import numpy as np
import pandas as pd
from tqdm import tqdm


from clinical_notes_extraction.config import PROJECT_ROOT
from clinical_notes_extraction.utils.llm.llm_extraction import ExtractionRunner

## Configuration

`ENVIRONMENT` selects the phase/environment data:
- Use `DEV` for model/strategy selection. We should use `dev data` to test and choose the best model/strategy selection; 
- Switch to `PROD` only for the final one-shot run of the winning combination (see the last section).

In [ ]:
# Phase: "train_val" for selection, "test" for the final held-out run.
ENVIRONMENT = "dev"

# Notebooks live in notebooks/, so the project root is the parent.
ROOT = Path(f"{PROJECT_ROOT}/scripts/3_information_extraction/3_2_medications_on_admission")
CONFIG_DIR = PROJECT_ROOT / "config"
PROMPTS_DIR = ROOT / "prompts"
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "data" / "llm_extraction_results" / ENVIRONMENT

STRATEGIES = ["zero_shot"] #, "few_shot", "dynamic"]

## Load configuration and split

`runnable_models.json` carries the runnable model tags and the Ollama block (`url`, `options`, `timeout_seconds`). 
The split is expected to hold `note_id`, `note_text`, and `cluster` — the last one drives the dynamic example lookup.

In [ ]:
config = json.loads(
    (CONFIG_DIR / "runnable_models.json").read_text(encoding="utf-8")
)
models = config["models"]
ollama_config = config["ollama"]

env_df = pd.read_parquet(DATA_DIR / f"{ENVIRONMENT}_sample.parquet")
notes = env_df[["note_id", "text"]].to_dict("records")

print(f"{len(models)} models x {len(STRATEGIES)} strategies x "
      f"{len(notes)} notes = {len(models) * len(STRATEGIES) * len(notes)} cells")

## Load prompt assets once

Reading these here avoids re-reading the same files on every one of the grid cells. 
- `role.md` is the system prompt; 
- the strategy templates are the user prompt shells.

In [ ]:
# System prompt (persona) and the shared expected-output schema.
role = (PROMPTS_DIR / "role.md").read_text(encoding="utf-8")
expected_template = (PROMPTS_DIR / "expected_template.md").read_text(encoding="utf-8")

# One user-prompt shell per strategy, keyed by strategy name.
strategy_templates = {
    strategy: (PROMPTS_DIR / f"{strategy}.md").read_text(encoding="utf-8")
    for strategy in STRATEGIES
}

# Few-shot example: 2 curated population examples, external to the 32-note sample.
#few_shot_examples = json.loads(
#    (DATA_DIR / "annotations/few_shot_examples.json").read_text(encoding="utf-8")
#)

# Dynamic examples: one annotated medoid per cluster, keyed by cluster id.

medoids_df = pd.read_parquet(DATA_DIR / f"annotations/dynamic_prompts/dynamic_prompt_medoids_annotations.parquet")
medoids = medoids_df[["note_id", "text", "annotation_json", "embedding"]].to_dict("records")

## Example selection per strategy

The only thing that varies across strategies is which examples get injected into `{EXAMPLES}`. 
- `zero_shot` passes an empty string (the placeholder simply vanishes on `.replace()`); 
- `few_shot` passes the fixed pool; 
- `dynamic` passes the single medoid of the note's cluster.

`format_examples` renders the example dicts into a text block. It uses `json.dumps` for the expected output — literal braces are exactly why prompt injection uses `.replace()` and never `.format()`.

> Check the delimiter format below against your actual `few_shot.md` / `dynamic.md` so the in-prompt examples match the shell around them.

In [ ]:
def format_examples(items: list[dict]) -> str:
    """Render example dicts (text + gold medications) into a prompt block."""
    blocks = []
    for i, ex in enumerate(items, start=1):
        output = json.dumps(
            {"medications": ex["medications"]}, indent=2, ensure_ascii=False
        )
        blocks.append(
            f"<example {i}>\n"
            f"<note>\n{ex['text']}\n</note>\n"
            f"<output>\n{output}\n</output>\n"
            f"</example {i}>"
        )
    return "\n\n".join(blocks)


def examples_for(strategy: str, note: dict) -> str:
    """Return the formatted example block for one (strategy, note) cell."""
    if strategy == "zero_shot":
        return ""
    #if strategy == "few_shot":
    #    return format_examples(few_shot_pool)
    #if strategy == "dynamic":
        # The note's cluster (known from clustering) selects its medoid.
    #    return format_examples([medoids[str(note["cluster"])]])
    raise ValueError(f"Unknown strategy: {strategy}")

## Run the grid

Every `(model, strategy, note)` cell goes through `extract_note`, which assembles the prompt, calls Ollama in JSON mode, validates with Pydantic and writes the record to disk. 
Because each cell caches its own result, rerunning this loop after an interruption only fills in what is missing.

In [ ]:
run_date = date.today().isoformat()

for model in models:
    for strategy in STRATEGIES:
        runner = ExtractionRunner(
            model=model,
            strategy=strategy,
            role=role,
            template=strategy_templates[strategy],
            expected_template=expected_template,
            results_dir=RESULTS_DIR,
            ollama_config=ollama_config,
            run_date=run_date,
            debug_prompt=True,
        )
        try:
            for note in tqdm(notes, desc=f"{model} | {strategy}"):
                runner.extract(
                    note_id=note["note_id"],
                    note_text=note["text"],
                    examples=examples_for(strategy, note),
                )
        finally:
            runner.unload()